In [ ]:
# セル0: GPU確認
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory // 1024 // 1024} MiB')

In [ ]:
# セル1: Google Drive マウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# セル2: リポジトリクローン
import os
if not os.path.exists('/content/jp-natural-voice-app'):
    !git clone https://github.com/stangler/jp-natural-voice-app /content/jp-natural-voice-app
else:
    print('Already cloned')
os.chdir('/content/jp-natural-voice-app')

In [ ]:
# セル3: パッケージセットアップ（固定バージョン）
!pip install torch==2.4.0+cu124 torchaudio==2.4.0+cu124 --index-url https://download.pytorch.org/whl/cu124 -q
!pip install transformers==4.37.2 huggingface_hub==0.19.4 numpy==1.26.4 -q
!pip install -r requirements.txt -q
print('パッケージセットアップ完了 ✅')

In [ ]:
# セル3.5: 【永続パッチ】jax/transformers 競合回避（セル3実行後は毎回実行）
filepath = '/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py'
with open(filepath, 'r') as f:
    lines = f.readlines()
new_lines = []
for line in lines:
    if 'import jax.numpy as jnp' in line:
        line = line.replace('import jax.numpy as jnp', 'pass  # jax disabled')
    new_lines.append(line)
with open(filepath, 'w') as f:
    f.writelines(new_lines)
print('jaxパッチ適用済み ✅')

In [ ]:
# セル4: bert_feature.py パッチ
# word2ph と text の長さ不一致による assert エラーを trim/pad で回避
patch_code = """
path = 'style_bert_vits2/nlp/japanese/bert_feature.py'
with open(path, 'r') as f:
    content = f.read()

old = '    text = text.replace(\"。\", \".\").replace(\"、\", \",\")\\n    assert len(word2ph) == len(text) + 2, text'

new = '''    text = text.replace(\"。\", \".\").replace(\"、\", \",\")
    expected = len(text) + 2
    if len(word2ph) != expected:
        if len(word2ph) > expected:
            word2ph = word2ph[:expected]
        else:
            word2ph = word2ph + [1] * (expected - len(word2ph))'''

count = content.count(old)
print(f'Found {count} occurrence(s)')
if count > 0:
    content = content.replace(old, new)
    with open(path, 'w') as f:
        f.write(content)
    print('bert_feature.py: Patched!')
else:
    print('bert_feature.py: Pattern not found (already patched or changed)')
"""

import subprocess
result = subprocess.run(['python3', '-c', patch_code], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)

In [ ]:
# セル5: lightning_fabric パッチ（毎回適用）
import importlib.util
if importlib.util.find_spec('lightning_fabric') is not None:
    import lightning_fabric
    lf_path = lightning_fabric.__file__.replace('__init__.py', 'utilities/cloud_io.py')
    with open(lf_path, 'r') as f:
        content = f.read()
    old = 'weights_only=True'
    new = 'weights_only=False'
    if old in content:
        content = content.replace(old, new)
        with open(lf_path, 'w') as f:
            f.write(content)
        print('lightning_fabric: Patched!')
    else:
        print('lightning_fabric: Already patched or not needed')
else:
    print('lightning_fabric not found - skip')

In [ ]:
# セル6: Drive からデータコピー
import os

# Drive の音声データ確認
DRIVE_WAV_PATH = '/content/drive/MyDrive/StyleBertVITS2/hirakawa_sample'
wav_count = !find {DRIVE_WAV_PATH}/wavs -name '*.wav' 2>/dev/null | wc -l
esd_exists = os.path.exists(f'{DRIVE_WAV_PATH}/esd.list')
print(f'音声ファイル数: {wav_count[0]}')
print(f'esd.list: {esd_exists}')

# Data ディレクトリにコピー
os.makedirs('Data/hirakawa_sample/wavs', exist_ok=True)
!cp -n {DRIVE_WAV_PATH}/wavs/*.wav Data/hirakawa_sample/wavs/ 2>/dev/null || true
!cp -n {DRIVE_WAV_PATH}/esd.list Data/hirakawa_sample/esd.list 2>/dev/null || true

copied = !find Data/hirakawa_sample/wavs -name '*.wav' | wc -l
print(f'コピー済み wav: {copied[0]} ファイル ✅')

# Drive に前処理済みファイルがある場合はコピー（スキップ可）
DRIVE_BERT_PATH = '/content/drive/MyDrive/StyleBertVITS2/hirakawa_sample_preprocessed'
if os.path.exists(DRIVE_BERT_PATH):
    print('前処理済みファイルが Drive に見つかりました。コピーします...')
    !cp {DRIVE_BERT_PATH}/*.bert.pt Data/hirakawa_sample/wavs/ 2>/dev/null || true
    !cp {DRIVE_BERT_PATH}/*.npy Data/hirakawa_sample/wavs/ 2>/dev/null || true
    bert_count = !find Data/hirakawa_sample/ -name '*.bert.pt' | wc -l
    npy_count = !find Data/hirakawa_sample/ -name '*.npy' | wc -l
    print(f'.bert.pt: {bert_count[0]}, .npy: {npy_count[0]}')
else:
    print('前処理済みファイルなし → 次のセルで前処理を実行してください')

In [ ]:
# セル7: config.json 生成
import os
os.chdir('/content/jp-natural-voice-app')

!python preprocess_all.py \
    --model_name hirakawa_sample \
    --batch_size 4 \
    --epochs 100 \
    --use_jp_extra

print('config.json 生成確認:', os.path.exists('Data/hirakawa_sample/config.json'))

In [ ]:
# セル7.5: esd.list のファイル名をフルパスに変換
esd_path = 'Data/hirakawa_sample/esd.list'

with open(esd_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

# すでにフルパス化されているかチェック
if not lines[0].startswith('Data/'):
    new_lines = []
    for line in lines:
        parts = line.strip().split('|')
        if len(parts) >= 1:
            parts[0] = f'Data/hirakawa_sample/wavs/{parts[0]}'
            new_lines.append('|'.join(parts) + '\n')
    with open(esd_path, 'w', encoding='utf-8') as f:
        f.writelines(new_lines)
    print(f'フルパス化完了: {len(new_lines)} 件 ✅')
else:
    print('すでにフルパス済み ✅')

print('先頭1行:', open(esd_path).readline().strip())

In [ ]:
# セル7.6: preprocess_text.py 実行（音素変換 → train.list / val.list 生成）
!python preprocess_text.py \
    --config-path Data/hirakawa_sample/config.json \
    --transcription-path Data/hirakawa_sample/esd.list \
    --cleaned-path Data/hirakawa_sample/esd.list.cleaned \
    --train-path Data/hirakawa_sample/train.list \
    --val-path Data/hirakawa_sample/val.list \
    --use_jp_extra

# 確認
with open('Data/hirakawa_sample/train.list', 'r', encoding='utf-8') as f:
    line = f.readline()
print('フィールド数:', len(line.strip().split('|')))
print('先頭1行:', line[:100])

In [ ]:
# セル7.7: config.json 設定（batch_size / fp16 / save_every_steps）
import json

config_path = 'Data/hirakawa_sample/config.json'
with open(config_path, 'r') as f:
    config = json.load(f)

config['train']['batch_size'] = 4
config['train']['fp16_run'] = True
config['train']['save_every_steps'] = 200

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(f"batch_size: {config['train']['batch_size']}")
print(f"fp16_run: {config['train']['fp16_run']}")
print(f"save_every_steps: {config['train']['save_every_steps']}")
print('config.json 更新完了 ✅')

In [ ]:
# セル8: 前処理（bert_gen / style_gen）
import subprocess

# bert_gen
bert_count = int(subprocess.check_output(
    'find Data/hirakawa_sample/ -name "*.bert.pt" | wc -l', shell=True
).decode().strip())
print(f'既存の .bert.pt: {bert_count} ファイル')

if bert_count < 180:
    print('bert_gen.py を実行します...')
    !python bert_gen.py --config Data/hirakawa_sample/config.json
else:
    print('✅ bert_gen 前処理済み → スキップ')

# style_gen（torchvision 競合回避のためアンインストール）
npy_count = int(subprocess.check_output(
    'find Data/hirakawa_sample/ -name "*.npy" | wc -l', shell=True
).decode().strip())
print(f'既存の .npy: {npy_count} ファイル')

if npy_count < 180:
    print('style_gen.py を実行します...')
    !pip uninstall torchvision -y -q
    !python style_gen.py --config Data/hirakawa_sample/config.json
else:
    print('✅ style_gen 前処理済み → スキップ')

# 最終確認
bert_count = int(subprocess.check_output('find Data/hirakawa_sample/ -name "*.bert.pt" | wc -l', shell=True).decode().strip())
npy_count = int(subprocess.check_output('find Data/hirakawa_sample/ -name "*.npy" | wc -l', shell=True).decode().strip())
print(f'.bert.pt: {bert_count} ファイル')
print(f'.npy:     {npy_count} ファイル')

In [ ]:
# セル9: 学習実行
import os
os.environ['MKL_THREADING_LAYER'] = 'GNU'

!python train_ms_jp_extra.py \
    -c Data/hirakawa_sample/config.json \
    -m Data/hirakawa_sample

In [ ]:
# セル10: モデルを Drive にバックアップ
import os

BACKUP_PATH = '/content/drive/MyDrive/StyleBertVITS2/hirakawa_sample_models'
os.makedirs(BACKUP_PATH, exist_ok=True)

!cp Data/hirakawa_sample/models/*.safetensors {BACKUP_PATH}/ 2>/dev/null || true
!cp Data/hirakawa_sample/config.json {BACKUP_PATH}/ 2>/dev/null || true

backed_up = os.listdir(BACKUP_PATH)
print(f'バックアップ完了: {backed_up} ✅')

In [ ]:
# セル11: 推論テスト
import subprocess

# 最新モデルを確認
models = !find Data/hirakawa_sample/models -name 'G_*.safetensors' | sort -V | tail -1
latest_model = models[0] if models else None
print(f'最新モデル: {latest_model}')

if latest_model:
    !python infer.py \
        --model_path {latest_model} \
        --config_path Data/hirakawa_sample/config.json \
        --text 'こんにちは、テストです。' \
        --output output_test.wav
    print('推論完了 ✅ → output_test.wav')
else:
    print('モデルが見つかりません。学習を完了してください。')